In [ ]:
#Programming Exercise 9-1 MuZero-Style Network SOLUTION

In [ ]:
# Cell 1: Imports
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random

In [ ]:
# Cell 3: MuZero-Style Network
class MuZeroNet(nn.Module):
    def __init__(self, state_size, action_size, hidden_size=64):
        super().__init__()

        # Representation function
        self.representation = nn.Sequential(
            nn.Linear(state_size, hidden_size),
            nn.ReLU()
        )

        # Dynamics function
        self.dynamics = nn.Sequential(
            nn.Linear(hidden_size + action_size, hidden_size),
            nn.ReLU()
        )

        self.reward_head = nn.Linear(hidden_size, 1)

        # Prediction function
        self.policy_head = nn.Linear(hidden_size, action_size)
        self.value_head = nn.Linear(hidden_size, 1)

    def initial_inference(self, state):
        hidden = self.representation(state)
        policy = self.policy_head(hidden)
        value = self.value_head(hidden)
        return hidden, policy, value

    def recurrent_inference(self, hidden, action_onehot):
        x = torch.cat([hidden, action_onehot], dim=1)
        next_hidden = self.dynamics(x)
        reward = self.reward_head(next_hidden)
        policy = self.policy_head(next_hidden)
        value = self.value_head(next_hidden)
        return next_hidden, reward, policy, value

In [ ]:
# Cell 4: Training Setup
env = GridWorld()
state_size = 25
action_size = 4

model = MuZeroNet(state_size, action_size)
optimizer = optim.Adam(model.parameters(), lr=0.001)

def one_hot(action, size):
    vec = np.zeros(size)
    vec[action] = 1
    return vec

In [ ]:
# Cell 5: Training Loop
episodes = 50

for ep in range(episodes):
    state = env.reset()
    state = torch.FloatTensor(state).unsqueeze(0)

    total_reward = 0

    for step in range(50):
        hidden, policy, value = model.initial_inference(state)

        action = np.random.choice(4)  # random exploration
        next_state, reward, done = env.step(action)

        next_state_t = torch.FloatTensor(next_state).unsqueeze(0)
        action_oh = torch.FloatTensor(one_hot(action, 4)).unsqueeze(0)

        next_hidden, pred_reward, pred_policy, pred_value = model.recurrent_inference(hidden, action_oh)

        # Targets
        target_reward = torch.tensor([[reward]], dtype=torch.float32)
        target_value = torch.tensor([[reward]], dtype=torch.float32)

        # Loss
        loss = (
            nn.MSELoss()(pred_reward, target_reward) +
            nn.MSELoss()(pred_value, target_value)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        state = next_state_t
        total_reward += reward

        if done:
            break

    print(f"Episode {ep+1}: Total Reward = {total_reward:.2f}")

Episode 1: Total Reward = 0.63
Episode 2: Total Reward = -0.50
Episode 3: Total Reward = -0.50
Episode 4: Total Reward = 0.60
Episode 5: Total Reward = 0.54
Episode 6: Total Reward = -0.50
Episode 7: Total Reward = 0.75
Episode 8: Total Reward = -0.50
Episode 9: Total Reward = -0.50
Episode 10: Total Reward = -0.50
Episode 11: Total Reward = -0.50
Episode 12: Total Reward = 0.84
Episode 13: Total Reward = -0.50
Episode 14: Total Reward = -0.50
Episode 15: Total Reward = 0.84
Episode 16: Total Reward = 0.57
Episode 17: Total Reward = -0.50
Episode 18: Total Reward = 0.69
Episode 19: Total Reward = -0.50
Episode 20: Total Reward = 0.92
Episode 21: Total Reward = 0.71
Episode 22: Total Reward = 0.60
Episode 23: Total Reward = -0.50
Episode 24: Total Reward = 0.63
Episode 25: Total Reward = -0.50
Episode 26: Total Reward = -0.50
Episode 27: Total Reward = -0.50
Episode 28: Total Reward = 0.54
Episode 29: Total Reward = -0.50
Episode 30: Total Reward = -0.50
Episode 31: Total Reward = -0.50

In [ ]:
# Cell 6: Inspect Learned Behavior
state = env.reset()
state = torch.FloatTensor(state).unsqueeze(0)

# Initial inference (no action yet)
hidden, policy, value = model.initial_inference(state)

print("=== INITIAL PREDICTION (Before Taking Action) ===")
print("Policy (action probabilities):")
print(torch.softmax(policy, dim=1).detach().numpy())

print("\n[KEY IDEA] The model predicts VALUE, not just actions:")
print(f"Estimated Value of Current State: {value.item():.4f}")

# Take a sample action to demonstrate dynamics
action = np.argmax(torch.softmax(policy, dim=1).detach().numpy())
action_oh = torch.FloatTensor(one_hot(action, 4)).unsqueeze(0)

next_hidden, pred_reward, next_policy, next_value = model.recurrent_inference(hidden, action_oh)

print("\n=== AFTER TAKING ACTION (Using Dynamics Function) ===")
print(f"Chosen Action: {action}")

print("\n[KEY IDEA] The model predicts REWARD from the action:")
print(f"Predicted Reward: {pred_reward.item():.4f}")

print("\n[KEY IDEA] The dynamics function simulates the next state internally:")
print("Next Policy (from imagined next state):")
print(torch.softmax(next_policy, dim=1).detach().numpy())

print("\nNext State Value Estimate:")
print(f"{next_value.item():.4f}")

print("\n=== LEARNING PROGRESS NOTE ===")
print("[KEY IDEA] Over time, total rewards printed during training should improve.")
print("This indicates the model is learning the environment structure.")

=== INITIAL PREDICTION (Before Taking Action) ===
Policy (action probabilities):
[[0.241293   0.26644114 0.26769346 0.22457244]]

[KEY IDEA] The model predicts VALUE, not just actions:
Estimated Value of Current State: -0.0789

=== AFTER TAKING ACTION (Using Dynamics Function) ===
Chosen Action: 2

[KEY IDEA] The model predicts REWARD from the action:
Predicted Reward: -0.0230

[KEY IDEA] The dynamics function simulates the next state internally:
Next Policy (from imagined next state):
[[0.23137526 0.27835676 0.25913164 0.2311364 ]]

Next State Value Estimate:
-0.0362

=== LEARNING PROGRESS NOTE ===
[KEY IDEA] Over time, total rewards printed during training should improve.
This indicates the model is learning the environment structure.
